# Pipeline Unificado — Sistema Híbrido de Omni-Ensemble (SEE)

Executor fino: a lógica vive em `src/`. Rode as células de cima para baixo.

**Se for a primeira execução numa máquina nova** (Codespaces/Colab/venv limpa), a célula de *Setup* abaixo instala as dependências automaticamente.

## Setup — raiz do projeto + dependências

In [1]:
# Bootstrap: acha a raiz do projeto (pasta que contém src/) e ajusta path/CWD
import os, sys
root = os.getcwd()
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'src')):
    root = os.path.dirname(root)
assert os.path.isdir(os.path.join(root, 'src')), 'pasta src/ nao encontrada'
os.chdir(root); sys.path.insert(0, root)
print('Raiz do projeto:', root)

Raiz do projeto: /workspaces/rp_hybrid_system_with_ensemble


In [2]:
# Instala dependencias se faltarem (seguro reexecutar). Precisa de internet.
import importlib.util, subprocess, sys
_need = {'sklearn': 'scikit-learn', 'numpy': 'numpy', 'scipy': 'scipy',
         'pandas': 'pandas', 'matplotlib': 'matplotlib', 'joblib': 'joblib'}
_missing = [pkg for mod, pkg in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print('Instalando:', _missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=True)
    print('Dependencias instaladas.')
else:
    print('Todas as dependencias ja presentes.')

Instalando: ['scikit-learn', 'joblib']
Dependencias instaladas.


In [3]:
import numpy as np
import pandas as pd

from src.dataset_loader import run_pipeline, datasets_with_pool, load_pool_arrays, load_xy
from src.train_pool import run_train_pool, build_pool
from src.ses_ga_single import run_ses_ga_pipeline
from src.ses_ga_multi import run_ses_ga_multi_pipeline
from src.des_dynamic import DynamicEnsembleSelector
from src.combination import combine_mean
from src.evaluator import evaluate_all, smape, mre, mase, nse, cod
from src.statistical_tests import friedman_test, bonferroni_dunn_test, plot_cd_diagram
print('Imports OK. Pool base:', list(build_pool().keys()))

Imports OK. Pool base: ['SVR', 'MLP', 'LR', 'Ridge', 'Lasso', 'ElasticNet', 'kNN', 'DT']


## T1 — Dados reais e pré-processamento

In [4]:
splits = run_pipeline()

  T1 — Carregamento e pré-processamento (dados reais, 5 bases)
[T1] finnish     alvo=Worksup  linhas 407→405 | features=37 | treino/teste=283/122  ⚠ 2 linha(s) com alvo<=0 descartada(s)
              descartadas: ['Case_Number', 'Project_tech_ID', 'YK', 'Project_name', 'Business_names', 'Protype_names', 'Hardware_names', 'Duration']
[T1] maxwell     alvo=Effort   linhas 62→62 | features=25 | treino/teste=43/19
              descartadas: ['Duration']
[T1] (pulada) china: arquivo ausente em data/raw/china.arff
[T1] (pulada) desharnais: arquivo ausente em data/raw/desharnais.csv
[T1] cocomo81    alvo=actual   linhas 63→63 | features=16 | treino/teste=44/19
----------------------------------------------------------------
[T1] ✓ 3/5 base(s) processada(s) → artefatos em data/processed/
[T1] Faltam (suba em data/raw/): ['china', 'desharnais']


## T2 — Pool de modelos base (sem ensembles)

In [5]:
pool = run_train_pool()

  T2 — Pool de modelos base (individuais, sem ensembles)
[T2] finnish     treinando 8 modelos... ok
[T2] maxwell     treinando 8 modelos... ok
[T2] cocomo81    treinando 8 modelos... ok
----------------------------------------------------------------
[T2] ✓ pool treinado em 3 base(s) (8 modelos) → matrizes em data/processed/


## T3 — SES-GA mono-objetivo

In [6]:
t3 = run_ses_ga_pipeline(verbose=False)
print({d: t3[d]['n_models_selected'] for d in t3}, '(# modelos selecionados por base)')

  T3 — SES-GA mono-objetivo (maximiza R²)

── FINNISH ──

── MAXWELL ──

── COCOMO81 ──

[T3] ✓ SES-GA em 3 base(s) → results/
{'finnish': 3, 'maxwell': 2, 'cocomo81': 1} (# modelos selecionados por base)


## T4 — SES-GA multi-objetivo (R² × parcimônia)

In [7]:
t4 = run_ses_ga_multi_pipeline(verbose=False)
print({d: t4[d]['best_balanced']['n_models'] for d in t4}, '(# modelos na solucao balanceada)')

  T4 — SES-GA multi-objetivo (R² × parcimônia) — NSGA-II

── FINNISH ──

── MAXWELL ──

── COCOMO81 ──

[T4] ✓ SES-GA-multi em 3 base(s) → results/
{'finnish': 2, 'maxwell': 1, 'cocomo81': 1} (# modelos na solucao balanceada)


## T5–T8 — Integração: DES, combinação, métricas e testes de hipótese

In [8]:
base_names = pool[next(iter(pool))]['model_names']
method_names = base_names + ['Static', 'SES-GA', 'SES-GA-Multi', 'DES']

summary_rows, matrix_rows = [], []
for d in datasets_with_pool():
    pred_train, pred_test, y_train, y_test = load_pool_arrays(d)
    X_train, X_test, _, _ = load_xy(d)

    des = DynamicEnsembleSelector(k=7, threshold=0.3)
    des.fit(X_train, pred_train, y_train)
    yp = {
        'DES':          des.predict(X_test, pred_test),
        'Static':       combine_mean(pred_test, None),
        'SES-GA':       combine_mean(pred_test, t3[d]['best_chromosome']),
        'SES-GA-Multi': combine_mean(pred_test, t4[d]['best_balanced']['chromosome']),
    }
    for name, pred in yp.items():
        summary_rows.append({'dataset': d, 'method': name,
                             'sMAPE': smape(y_test, pred), 'MRE': mre(y_test, pred),
                             'MASE': mase(y_test, pred), 'NSE': nse(y_test, pred),
                             'COD': cod(y_test, pred)})
    base_err = [smape(y_test, pred_test[:, i]) for i in range(pred_test.shape[1])]
    matrix_rows.append(base_err + [smape(y_test, yp['Static']), smape(y_test, yp['SES-GA']),
                                   smape(y_test, yp['SES-GA-Multi']), smape(y_test, yp['DES'])])

summary_df = pd.DataFrame(summary_rows)
display(summary_df.pivot(index='dataset', columns='method', values='sMAPE').round(3))

results_matrix = np.array(matrix_rows, dtype=float)
print('Matriz de erro sMAPE (bases x metodos):', results_matrix.shape)
display(pd.DataFrame(results_matrix, columns=method_names, index=datasets_with_pool()).round(2))

[DES] fit() | 283 instâncias treino | 8 modelos no pool
[DES] predict() | 122 instâncias preditas | k=7 | top=30% modelos
[DES] fit() | 43 instâncias treino | 8 modelos no pool
[DES] predict() | 19 instâncias preditas | k=7 | top=30% modelos
[DES] fit() | 44 instâncias treino | 8 modelos no pool
[DES] predict() | 19 instâncias preditas | k=7 | top=30% modelos


method,DES,SES-GA,SES-GA-Multi,Static
dataset,,,,
cocomo81,114.500,93.015,93.015,127.218
finnish,61.209,65.007,61.595,61.719
maxwell,78.367,74.691,82.218,70.568


Matriz de erro sMAPE (bases x metodos): (3, 12)


,SVR,MLP,LR,Ridge,Lasso,ElasticNet,kNN,DT,Static,SES-GA,SES-GA-Multi,DES
finnish,79.32,65.97,78.92,71.20,63.53,68.44,64.50,68.06,61.72,65.01,61.59,61.21
maxwell,76.06,67.70,68.74,79.36,65.68,66.12,72.30,82.22,70.57,74.69,82.22,78.37
cocomo81,136.47,159.30,144.90,132.66,123.34,138.17,101.16,93.02,127.22,93.02,93.02,114.50


In [9]:
control = 'SES-GA-Multi'  # metodo proposto (na Etapa 1: + combinacao por competencia)
fr = friedman_test(results_matrix, method_names)
if fr['significant']:
    bonferroni_dunn_test(fr, control_model=control, n_datasets=results_matrix.shape[0])
else:
    print('[INFO] Friedman nao significativo — post-hoc nao aplicado.')
plot_cd_diagram(fr, control_model=control, n_datasets=results_matrix.shape[0])


  TESTE DE FRIEDMAN
  Estatística de Friedman : 9.5641
  p-value                 : 0.569969
  Significativo (p<0.05)  : ❌ NÃO

  Ranks médios por modelo:
    Lasso                rank=3.667  ███████████
    kNN                  rank=5.000  ███████████████
    Static               rank=5.000  ███████████████
    SES-GA               rank=5.000  ███████████████
    DES                  rank=5.000  ███████████████
    SES-GA-Multi         rank=5.667  █████████████████
    DT                   rank=6.667  ████████████████████
    ElasticNet           rank=7.000  █████████████████████
    MLP                  rank=7.333  ██████████████████████
    LR                   rank=8.667  ██████████████████████████
    Ridge                rank=9.333  ████████████████████████████
    SVR                  rank=9.667  █████████████████████████████
[INFO] Friedman nao significativo — post-hoc nao aplicado.

[PLOT] Diagrama salvo em: results/cd_diagram.png
